# 🎬 YouTube Viral Video Clipper para Redes Sociais

Este notebook automatiza:
1. **Monitoramento** de canais YouTube por views e viralização
2. **Histórico** de vídeos processados (evita duplicação)
3. **Download** de vídeos
4. **IA** para identificar momentos mais assistidos
5. **Corte** inteligente (5s antes + 5s depois do pico)
6. **Conversão** para formatos TikTok/Instagram/Facebook (9:16 vertical)

## 📦 Instalação das Dependências

In [ ]:
!pip install -q youtube-transcript-api pytube moviepy torch torchvision transformers pillow numpy pandas google-auth-oauthlib google-api-python-client

## 🔑 Configuração da API do YouTube

Você precisará de uma API Key do YouTube Data API v3:
1. Acesse https://console.cloud.google.com/
2. Crie um projeto e ative a YouTube Data API v3
3. Gere uma API Key

In [ ]:
# Configure sua API Key aqui
YOUTUBE_API_KEY = "SUA_API_KEY_AQUI"

# Lista de canais para monitorar (IDs dos canais)
CHANNEL_IDS = [
    "UCX6OQ3DkcsbYNE6H8uQQuVA",  # Exemplo: MrBeast
    # Adicione mais IDs de canais aqui
]

# Limiares para considerar vídeo viral
MIN_VIEWS = 100000  # Views mínimas
MIN_LIKE_RATIO = 0.95  # Ratio mínimo de likes/views
MAX_RESULTS_PER_CHANNEL = 5  # Máximo de vídeos por canal por execução

## 💾 Sistema de Histórico

In [ ]:
import json
import os
from datetime import datetime

HISTORY_FILE = "processed_videos_history.json"

def load_history():
    """Carrega o histórico de vídeos processados"""
    if os.path.exists(HISTORY_FILE):
        with open(HISTORY_FILE, 'r') as f:
            return json.load(f)
    return []

def save_to_history(video_id, video_info):
    """Salva vídeo no histórico após processamento"""
    history = load_history()
    record = {
        "video_id": video_id,
        "processed_at": datetime.now().isoformat(),
        "title": video_info.get("title", ""),
        "views": video_info.get("views", 0),
        "clip_path": video_info.get("clip_path", "")
    }
    history.append(record)
    with open(HISTORY_FILE, 'w') as f:
        json.dump(history, f, indent=2)
    print(f"✅ Vídeo {video_id} salvo no histórico")

def is_video_processed(video_id):
    """Verifica se vídeo já foi processado"""
    history = load_history()
    return any(record["video_id"] == video_id for record in history)

print("Sistema de histórico inicializado!")

## 🔍 Monitoramento de Canais YouTube

In [ ]:
from googleapiclient.discovery import build
import pandas as pd

def get_youtube_client():
    """Inicializa cliente da API do YouTube"""
    return build('youtube', 'v3', developerKey=YOUTUBE_API_KEY)

def search_viral_videos(channel_ids, max_results=5):
    """
    Busca vídeos virais nos canais monitorados
    Retorna lista de vídeos ordenados por views/engajamento
    """
    youtube = get_youtube_client()
    viral_videos = []
    
    for channel_id in channel_ids:
        try:
            # Busca vídeos recentes do canal
            search_response = youtube.search().list(
                part='snippet',
                channelId=channel_id,
                order='viewCount',
                type='video',
                maxResults=max_results
            ).execute()
            
            video_ids = [item['id']['videoId'] for item in search_response['items']]
            
            # Detalhes completos dos vídeos
            videos_response = youtube.videos().list(
                part='snippet,statistics,contentDetails',
                id=','.join(video_ids)
            ).execute()
            
            for video in videos_response['items']:
                stats = video['statistics']
                views = int(stats.get('viewCount', 0))
                likes = int(stats.get('likeCount', 0))
                comments = int(stats.get('commentCount', 0))
                
                # Calcula score de viralização
                like_ratio = likes / views if views > 0 else 0
                engagement_score = (likes + comments) / views if views > 0 else 0
                
                if views >= MIN_VIEWS and like_ratio >= MIN_LIKE_RATIO:
                    viral_videos.append({
                        'video_id': video['id'],
                        'title': video['snippet']['title'],
                        'channel': video['snippet']['channelTitle'],
                        'views': views,
                        'likes': likes,
                        'comments': comments,
                        'like_ratio': like_ratio,
                        'engagement_score': engagement_score,
                        'duration': video['contentDetails']['duration'],
                        'published_at': video['snippet']['publishedAt']
                    })
        except Exception as e:
            print(f"Erro ao processar canal {channel_id}: {e}")
    
    # Ordena por score de engajamento
    viral_videos.sort(key=lambda x: x['engagement_score'], reverse=True)
    
    return viral_videos

print("Função de monitoramento pronta!")

## 📥 Download de Vídeos

In [ ]:
from pytube import YouTube
import os

def download_video(video_id, output_path="downloads"):
    """
    Baixa vídeo do YouTube
    Retorna caminho do arquivo baixado
    """
    os.makedirs(output_path, exist_ok=True)
    
    url = f"https://www.youtube.com/watch?v={video_id}"
    yt = YouTube(url)
    
    # Seleciona stream com melhor qualidade
    stream = yt.streams.filter(progressive=True, file_extension='mp4').order_by('resolution').desc().first()
    
    if not stream:
        stream = yt.streams.get_highest_resolution()
    
    filename = f"{video_id}.mp4"
    filepath = os.path.join(output_path, filename)
    
    print(f"📥 Baixando: {yt.title}")
    stream.download(output_path=output_path, filename=filename)
    print(f"✅ Download concluído: {filepath}")
    
    return filepath, yt.title

print("Função de download pronta!")

## 🤖 IA para Identificar Momentos Virais

In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForVideoClassification
from moviepy.editor import VideoFileClip
import numpy as np
from PIL import Image

class ViralMomentDetector:
    """
    Detecta momentos virais em vídeos usando múltiplas estratégias:
    1. Análise de áudio (picos de volume/emoção)
    2. Análise visual (mudanças de cena, movimento)
    3. Modelo de IA para classificação de conteúdo
    """
    
    def __init__(self):
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        print(f"Usando dispositivo: {self.device}")
        
    def extract_audio_peaks(self, video_path, window_size=5):
        """
        Extrai picos de áudio (momentos de maior volume/energia)
        Retorna timestamps dos picos
        """
        clip = VideoFileClip(video_path)
        duration = clip.duration
        
        # Amostra áudio em intervalos
        sample_rate = 1.0  # segundos entre amostras
        timestamps = []
        audio_levels = []
        
        t = 0
        while t < duration:
            # Extrai segmento de áudio
            segment = clip.subclip(t, min(t + sample_rate, duration))
            
            # Calcula nível de áudio (RMS)
            try:
                audio = segment.audio
                if audio:
                    # Escreve áudio temporário para análise
                    temp_audio = f"/tmp/audio_{t}.wav"
                    audio.write_audiofile(temp_audio, verbose=False, logger=None)
                    
                    # Lê e calcula RMS
                    import wave
                    with wave.open(temp_audio, 'rb') as wf:
                        frames = wf.readframes(wf.getnframes())
                        audio_data = np.frombuffer(frames, dtype=np.int16)
                        rms = np.sqrt(np.mean(audio_data.astype(float)**2))
                        audio_levels.append((t, rms))
                    
                    os.remove(temp_audio)
            except:
                pass
            
            t += sample_rate
        
        clip.close()
        
        # Encontra picos
        if len(audio_levels) > 0:
            audio_levels.sort(key=lambda x: x[1], reverse=True)
            return [ts for ts, _ in audio_levels[:10]]  # Top 10 picos
        
        return [duration / 2]  # Fallback: meio do vídeo
    
    def detect_scene_changes(self, video_path, threshold=0.3):
        """
        Detecta mudanças de cena (transições importantes)
        """
        clip = VideoFileClip(video_path)
        duration = clip.duration
        
        frame_times = []
        prev_frame = None
        
        # Amostra frames a cada 2 segundos
        t = 0
        while t < duration:
            try:
                frame = clip.get_frame(t)
                frame_gray = np.mean(frame, axis=2)
                
                if prev_frame is not None:
                    # Calcula diferença entre frames
                    diff = np.mean(np.abs(frame_gray.astype(float) - prev_frame.astype(float)))
                    normalized_diff = diff / 255.0
                    
                    if normalized_diff > threshold:
                        frame_times.append(t)
                
                prev_frame = frame_gray
            except:
                pass
            
            t += 2
        
        clip.close()
        
        return frame_times if frame_times else [duration / 2]
    
    def find_best_moment(self, video_path):
        """
        Combina múltiplas estratégias para encontrar o melhor momento
        Retorna timestamp ideal para corte
        """
        print("🔍 Analisando vídeo para encontrar momento viral...")
        
        # Estratégia 1: Picos de áudio
        audio_peaks = self.extract_audio_peaks(video_path)
        print(f"   - Picos de áudio encontrados: {len(audio_peaks)}")
        
        # Estratégia 2: Mudanças de cena
        scene_changes = self.detect_scene_changes(video_path)
        print(f"   - Mudanças de cena encontradas: {len(scene_changes)}")
        
        # Combina resultados (prioriza picos de áudio)
        candidate_timestamps = audio_peaks[:3] + scene_changes[:2]
        
        if not candidate_timestamps:
            # Fallback: usa meio do vídeo
            clip = VideoFileClip(video_path)
            best_timestamp = clip.duration / 2
            clip.close()
        else:
            # Escolhe timestamp que aparece mais cedo (geralmente hook inicial)
            best_timestamp = min(candidate_timestamps)
        
        print(f"   ⭐ Melhor momento encontrado: {best_timestamp:.2f}s")
        return best_timestamp

print("Detector de momentos virais pronto!")

## ✂️ Corte Inteligente de Vídeo

In [ ]:
from moviepy.editor import VideoFileClip
import os

def cut_video_clip(video_path, center_timestamp, output_path="clips", clip_duration=10):
    """
    Corta vídeo centrado no timestamp especificado
    Inclui 5s antes e 5s depois (total 10s)
    """
    os.makedirs(output_path, exist_ok=True)
    
    clip = VideoFileClip(video_path)
    duration = clip.duration
    
    # Calcula início e fim do corte
    half_duration = clip_duration / 2
    start_time = max(0, center_timestamp - half_duration)
    end_time = min(duration, center_timestamp + half_duration)
    
    # Ajusta se estiver nas bordas
    if start_time == 0:
        end_time = min(duration, clip_duration)
    elif end_time == duration:
        start_time = max(0, duration - clip_duration)
    
    # Extrai clip
    final_clip = clip.subclip(start_time, end_time)
    
    # Gera nome do arquivo
    video_name = os.path.basename(video_path).replace('.mp4', '')
    output_filename = f"{video_name}_clip_{int(start_time)}-{int(end_time)}.mp4"
    output_filepath = os.path.join(output_path, output_filename)
    
    print(f"✂️ Cortando vídeo: {start_time:.2f}s até {end_time:.2f}s")
    final_clip.write_videofile(
        output_filepath,
        codec='libx264',
        audio_codec='aac',
        temp_audiofile='/tmp/temp-audio.m4a',
        remove_temp=True,
        verbose=False,
        logger=None
    )
    
    clip.close()
    final_clip.close()
    
    print(f"✅ Clip salvo: {output_filepath}")
    return output_filepath

print("Função de corte pronta!")

## 📱 Conversão para Formato Vertical (TikTok/Instagram/Facebook)

In [ ]:
from moviepy.editor import VideoFileClip
import os

def convert_to_vertical(video_path, output_path="social_clips", platform="tiktok"):
    """
    Converte vídeo para formato vertical 9:16
    Plataformas suportadas: tiktok, instagram_reels, facebook_reels
    """
    os.makedirs(output_path, exist_ok=True)
    
    clip = VideoFileClip(video_path)
    width, height = clip.size
    duration = clip.duration
    
    # Dimensões alvo (9:16)
    target_width = 1080
    target_height = 1920
    
    print(f"📱 Convertendo para formato vertical ({target_width}x{target_height})...")
    
    # Calcula crop central
    target_ratio = target_width / target_height
    current_ratio = width / height
    
    if current_ratio > target_ratio:
        # Vídeo é mais largo que alto - crop horizontal
        new_width = int(height * target_ratio)
        x_center = width / 2
        x1 = x_center - new_width / 2
        x2 = x_center + new_width / 2
        cropped_clip = clip.crop(x1=x1, y1=0, x2=x2, y2=height)
    else:
        # Vídeo é mais alto que largo - crop vertical
        new_height = int(width / target_ratio)
        y_center = height / 2
        y1 = y_center - new_height / 2
        y2 = y_center + new_height / 2
        cropped_clip = clip.crop(x1=0, y1=y1, x2=width, y2=y2)
    
    # Redimensiona para dimensões alvo
    resized_clip = cropped_clip.resize((target_width, target_height))
    
    # Gera nome do arquivo
    video_name = os.path.basename(video_path).replace('.mp4', '')
    output_filename = f"{video_name}_{platform}_vertical.mp4"
    output_filepath = os.path.join(output_path, output_filename)
    
    # Exporta vídeo
    resized_clip.write_videofile(
        output_filepath,
        codec='libx264',
        audio_codec='aac',
        fps=30,
        bitrate='5000k',
        temp_audiofile='/tmp/temp-audio-vertical.m4a',
        remove_temp=True,
        verbose=False,
        logger=None
    )
    
    clip.close()
    cropped_clip.close()
    resized_clip.close()
    
    print(f"✅ Vídeo vertical salvo: {output_filepath}")
    return output_filepath

print("Função de conversão vertical pronta!")

## 🚀 Pipeline Completo de Processamento

In [ ]:
def process_viral_video(video_info, detector):
    """
    Pipeline completo: download → análise IA → corte → conversão
    """
    video_id = video_info['video_id']
    
    print(f"\n{'='*60}")
    print(f"🎬 Processando: {video_info['title']}")
    print(f"📊 Views: {video_info['views']:,} | Engagement: {video_info['engagement_score']:.2%}")
    print(f"{'='*60}\n")
    
    # Step 1: Download
    video_path, title = download_video(video_id)
    
    # Step 2: Detectar momento viral com IA
    best_moment = detector.find_best_moment(video_path)
    
    # Step 3: Cortar vídeo (5s antes + 5s depois)
    clip_path = cut_video_clip(video_path, best_moment)
    
    # Step 4: Converter para formato vertical
    platforms = ['tiktok', 'instagram', 'facebook']
    vertical_paths = []
    
    for platform in platforms:
        vertical_path = convert_to_vertical(clip_path, platform=platform)
        vertical_paths.append(vertical_path)
    
    # Step 5: Salvar no histórico
    save_to_history(video_id, {
        "title": title,
        "views": video_info['views'],
        "clip_path": clip_path,
        "vertical_paths": vertical_paths
    })
    
    print(f"\n✅ Processo concluído!")
    print(f"📁 Clip original: {clip_path}")
    print(f"📱 Versões verticais:")
    for path in vertical_paths:
        print(f"   - {path}")
    
    return {
        'video_id': video_id,
        'original_clip': clip_path,
        'vertical_clips': vertical_paths
    }

print("Pipeline completo pronto!")

## ▶️ Execução Principal

In [ ]:
def main():
    """
    Função principal que executa todo o fluxo
    """
    print("🚀 Iniciando monitoramento de vídeos virais...\n")
    
    # Inicializa detector de momentos virais
    detector = ViralMomentDetector()
    
    # Busca vídeos virais nos canais monitorados
    viral_videos = search_viral_videos(CHANNEL_IDS, MAX_RESULTS_PER_CHANNEL)
    
    if not viral_videos:
        print("❌ Nenhum vídeo viral encontrado nos critérios definidos.")
        return
    
    print(f"📺 Encontrados {len(viral_videos)} vídeos virais potenciais\n")
    
    # Processa cada vídeo (que ainda não foi processado)
    processed_count = 0
    
    for video_info in viral_videos:
        video_id = video_info['video_id']
        
        # Verifica histórico
        if is_video_processed(video_id):
            print(f"⏭️  Pulando vídeo já processado: {video_info['title']}")
            continue
        
        # Processa vídeo
        try:
            result = process_viral_video(video_info, detector)
            processed_count += 1
            
            # Limita número de processamentos por execução
            if processed_count >= 3:
                print("\n⚠️  Limite de 3 vídeos atingido nesta execução.")
                break
        except Exception as e:
            print(f"❌ Erro ao processar vídeo {video_id}: {e}")
            continue
    
    print(f"\n{'='*60}")
    print(f"✅ Conclusão: {processed_count} vídeo(s) processado(s) com sucesso!")
    print(f"{'='*60}")
    
    # Mostra histórico
    history = load_history()
    print(f"\n📜 Histórico total: {len(history)} vídeo(s)")

# Executa o pipeline
if __name__ == "__main__":
    main()

## 📊 Visualizar Histórico de Processamento

In [ ]:
def view_history():
    """Exibe histórico formatado de vídeos processados"""
    history = load_history()
    
    if not history:
        print("📭 Histórico vazio.")
        return
    
    df = pd.DataFrame(history)
    df['processed_at'] = pd.to_datetime(df['processed_at'])
    df = df.sort_values('processed_at', ascending=False)
    
    print("\n" + "="*80)
    print("📜 HISTÓRICO DE VÍDEOS PROCESSADOS")
    print("="*80)
    
    for _, row in df.iterrows():
        print(f"\n🎬 {row['title']}")
        print(f"   ID: {row['video_id']}")
        print(f"   Views: {row['views']:,}")
        print(f"   Processado em: {row['processed_at'].strftime('%d/%m/%Y %H:%M')}")
        print(f"   Clip: {row.get('clip_path', 'N/A')}")
    
    print("\n" + "="*80)
    print(f"Total: {len(history)} vídeo(s)")
    print("="*80)

# Execute para ver o histórico
# view_history()

## 📝 Instruções de Uso

### 1. Configuração Inicial
- Obtenha uma API Key do YouTube Data API v3
- Substitua `YOUTUBE_API_KEY` pela sua chave
- Adicione os IDs dos canais que deseja monitorar em `CHANNEL_IDS`

### 2. Como Obter ID de Canal YouTube
- Acesse o canal no YouTube
- O ID está na URL ou em: https://www.youtube.com/account_advanced
- Ou use: https://commentpicker.com/youtube-channel-id.php

### 3. Executar
1. Execute todas as células em ordem (Runtime → Run all)
2. O script irá:
   - Buscar vídeos virais nos canais
   - Baixar os vídeos
   - Analisar com IA para encontrar momentos de pico
   - Cortar 5s antes e 5s depois do pico
   - Converter para formato vertical 9:16
   - Salvar no histórico

### 4. Arquivos Gerados
- `downloads/`: Vídeos originais baixados
- `clips/`: Clips cortados (10 segundos)
- `social_clips/`: Versões verticais para redes sociais
- `processed_videos_history.json`: Histórico de processamento

### 5. Personalização
- Ajuste `MIN_VIEWS` e `MIN_LIKE_RATIO` para filtrar vídeos
- Modifique `clip_duration` para alterar duração do corte
- Edite `CHANNEL_IDS` para monitorar outros canais

## ⚠️ Notas Importantes

1. **API do YouTube**: Requer quota da API (gratuita até 10.000 unidades/dia)
2. **Direitos Autorais**: Use apenas conteúdo que você tem direito ou para fair use
3. **Tempo de Processamento**: Vídeos longos podem demorar para analisar
4. **Armazenamento**: Delete arquivos antigos periodicamente no Google Drive
5. **GPU**: Ative GPU (Runtime → Change runtime type → GPU) para análise mais rápida